In [55]:
import pandas as pd
import numpy as np

print("Projet WC2026 ML lancé 🚀")


Projet WC2026 ML lancé 🚀


In [56]:
import pandas as pd
import numpy as np

# Charger le dataset
df = pd.read_csv("../data/results.csv")

# Afficher les 5 premières lignes
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [57]:
print(df.shape)

(49437, 9)


In [58]:
print(df.columns)

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral'],
      dtype='str')


In [59]:
df.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
5605,1962-09-22,Trinidad and Tobago,Suriname,1.0,1.0,Friendly,Port of Spain,Trinidad and Tobago,False
28737,2004-10-14,Taiwan,Palestine,0.0,1.0,FIFA World Cup qualification,Taipei,Taiwan,False
38037,2014-09-07,Denmark,Armenia,2.0,1.0,UEFA Euro qualification,Copenhagen,Denmark,False
46125,2023-06-18,Guatemala,Venezuela,0.0,1.0,Friendly,East Hartford,United States,True
18926,1993-04-18,Japan,United Arab Emirates,2.0,0.0,FIFA World Cup qualification,Tokyo,Japan,False


In [60]:
df = df[[
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament"
]]

In [61]:
df.sample(5)

,date,home_team,away_team,home_score,away_score,tournament
27559,2003-09-10,Serbia,Italy,1.0,1.0,UEFA Euro qualification
736,1922-06-11,Norway,France,7.0,0.0,Friendly
12671,1981-03-21,Saudi Arabia,Iraq,1.0,0.0,FIFA World Cup qualification
20929,1996-03-27,Poland,Slovenia,0.0,0.0,Friendly
14336,1984-10-10,German DR,Algeria,5.0,2.0,Friendly


In [62]:
def get_result(row):

    # Si l'équipe domicile marque plus
    if row["home_score"] > row["away_score"]:
        return 0

    # Si l'équipe extérieure marque plus
    elif row["home_score"] < row["away_score"]:
        return 2

    # Sinon match nul
    else:
        return 1

In [63]:
# Appliquer la fonction "get_result" à chaque ligne du DataFrame

In [64]:
df["result"] = df.apply(get_result, axis=1)

In [65]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,result
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,1
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,1
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,0


In [66]:
df["result"].value_counts()

result
0    24192
2    13948
1    11297
Name: count, dtype: int64

In [67]:
teams_stats = {}

In [68]:
for _, row in df.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    if home not in teams_stats:
        teams_stats[home] = {
            "matches": 0,
            "wins": 0
        }

    if away not in teams_stats:
        teams_stats[away] = {
            "matches": 0,
            "wins": 0
        }

    teams_stats[home]["matches"] += 1
    teams_stats[away]["matches"] += 1

    if row["result"] == 0:
        teams_stats[home]["wins"] += 1

    elif row["result"] == 2:
        teams_stats[away]["wins"] += 1

In [69]:
teams_stats["France"]

{'matches': 937, 'wins': 476}

In [70]:
for team in teams_stats:

    matches = teams_stats[team]["matches"]
    wins = teams_stats[team]["wins"]

    teams_stats[team]["win_rate"] = wins / matches

In [71]:
teams_stats["France"]

{'matches': 937, 'wins': 476, 'win_rate': 0.5080042689434365}

In [72]:
def get_team_win_rate(team):
    return teams_stats[team]["win_rate"]

In [73]:
df["home_win_rate"] = df["home_team"].apply(get_team_win_rate)

df["away_win_rate"] = df["away_team"].apply(get_team_win_rate)

In [74]:
df[[
    "home_team",
    "away_team",
    "home_win_rate",
    "away_win_rate",
    "result"
]].head()

,home_team,away_team,home_win_rate,away_win_rate,result
0,Scotland,England,0.470726,0.571429,1
1,England,Scotland,0.571429,0.470726,0
2,Scotland,England,0.470726,0.571429,0
3,England,Scotland,0.571429,0.470726,1
4,Scotland,England,0.470726,0.571429,0


In [75]:
X = df[[
    "home_win_rate",
    "away_win_rate"
]]

y = df["result"]

In [76]:
X.head()

,home_win_rate,away_win_rate
0,0.470726,0.571429
1,0.571429,0.470726
2,0.470726,0.571429
3,0.571429,0.470726
4,0.470726,0.571429


In [77]:
y.head()

0    1
1    0
2    0
3    1
4    0
Name: result, dtype: int64

In [78]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [79]:
print(X_train.shape)
print(X_test.shape)

(39549, 2)
(9888, 2)


In [80]:
from sklearn.linear_model import LogisticRegression

In [81]:
model = LogisticRegression(max_iter=1000)